In [7]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)


@tool
def get_weather(city: str) -> str:
    """도시의 날씨 정보를 조회합니다."""
    weather_data = {
        "서울": "현재 비가 내리고 있습니다. 기온은 18도입니다.",
        "부산": "현재 맑습니다. 기온은 24도입니다.",
        "대구": "현재 흐립니다. 기온은 21도입니다.",
    }

    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")


agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="""
                당신은 날씨 안내 Agent입니다.

                사용자가 날씨를 물어보면 반드시 weather tool을 사용하세요.

                날씨 정보를 확인한 뒤,
                사용자의 질문에 맞게 최종 답변을 작성하세요.
                """,
)


def agent_node(state: MessagesState):
    result = agent.invoke({"messages": state["messages"]})

    return {"messages": result["messages"]}


tool_node = ToolNode([get_weather])


def should_continue(state: MessagesState):
    last_messages = state["messages"][-1]

    if last_messages.tool_calls:
        return "tools"
    return "end"


graph = StateGraph(MessagesState)

graph.add_node("agent", agent_node)

graph.add_node("tools", tool_node)

graph.add_edge(START, "agent")

graph.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})

graph.add_edge("tools", "agent")

app = graph.compile()

result = app.invoke(
    {
        "messages": [
            {"role": "user", "content": "서울 날씨를 확인하고 우산이 필요한지 알려줘."}
        ]
    }
)

for message in result["messages"]:
    print("\n------------------------------")
    print(type(message).__name__)
    print("------------------------------")

    print(message.content)

    if getattr(message, "tool_calls", None):
        print("Tool Calls:")
        print(message.tool_calls)


------------------------------
HumanMessage
------------------------------
서울 날씨를 확인하고 우산이 필요한지 알려줘.

------------------------------
AIMessage
------------------------------

Tool Calls:
[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'chatcmpl-tool-b4c9a6933b59cd79', 'type': 'tool_call'}]

------------------------------
ToolMessage
------------------------------
현재 비가 내리고 있습니다. 기온은 18도입니다.

------------------------------
AIMessage
------------------------------
현재 서울은 비가 내리고 있으니 **우산이 필요합니다**. 18도이니 조금 쌀쌀하지만 비를 피하기 위해 우산을 준비해 주세요.


In [11]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv()
API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)


@tool
def search_product(product_name: str) -> str:
    """상품 이름으로 상품 정보를 검색합니다."""
    products = {
        "아이폰 17": {"name": "아이폰 17", "price": 150000},
        "아이폰 17 프로": {"name": "아이폰 17 프로", "price": 180000},
        "맥북 에어": {"name": "맥북 에어", "price": 160000},
    }

    product = products.get(product_name)

    if product is None:
        return f"{product_name} 상품을 찾을 수 없습니다."

    return f"상품명: {product['name']}\n가격: {product['price']}원"


@tool
def calculate_discount(price: int, discount_percent: float) -> str:
    """상품 가격과 할인율을 받아 할인된 가격을 계산합니다."""

    discount_amount = price * (discount_percent / 100)

    final_price = price - discount_amount

    return (
        f"원래 가격: {price:,.0f}원\n"
        f"할인율: {discount_percent}%\n"
        f"할인 금액: {discount_amount:,.0f}원\n"
        f"최종 가격: {final_price:,.0f}원"
    )


agent = create_agent(
    model=llm,
    tools=[search_product, calculate_discount],
    system_prompt="""
                    당신은 상품 가격을 안내하는 Agent입니다.

                    사용자가 상품 정보를 요청하면
                    search_product Tool을 사용하세요.

                    할인 가격이나 할인 금액을 계산해야 한다면
                    calculate_discount Tool을 사용하세요.

                    필요한 Tool을 순서대로 사용할 수 있습니다.

                    모든 필요한 정보를 확인한 뒤
                    최종 답변을 작성하세요.
                    """,
)


def agent_node(state: MessagesState):
    result = agent.invoke({"messages": state["messages"]})

    return {"messages": result["messages"]}


tool_node = ToolNode([search_product, calculate_discount])

graph = StateGraph(MessagesState)

graph.add_node("agent", agent_node)

graph.add_node("tools", tool_node)

graph.add_edge(START, "agent")

graph.add_conditional_edges("agent", tools_condition)

graph.add_edge("tools", "agent")

app = graph.compile()

result = app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": ("아이폰 17의 가격을 찾아서 10% 할인된 가격도 알려주세요."),
            }
        ]
    }
)


print("전체 실행 과정")
print("=" * 70)

for index, message in enumerate(result["messages"]):
    print(f"\n[{index}] {type(message).__name__}")

    if message.content:
        print("내용:")
        print(message.content)

    if getattr(message, "tool_calls", None):
        print("Tool Calls:")
        for tool_call in message.tool_calls:
            print(f"  - {tool_call['name']}")
            print(f"    args: {tool_call['args']}")


print("최종 답변")
print("=" * 70)

print(result["messages"][-1].content)

전체 실행 과정

[0] HumanMessage
내용:
아이폰 17의 가격을 찾아서 10% 할인된 가격도 알려주세요.

[1] AIMessage
Tool Calls:
  - search_product
    args: {'product_name': '아이폰 17'}

[2] ToolMessage
내용:
상품명: 아이폰 17
가격: 150000원

[3] AIMessage
Tool Calls:
  - calculate_discount
    args: {'price': 150000, 'discount_percent': 10}

[4] ToolMessage
내용:
원래 가격: 150,000원
할인율: 10.0%
할인 금액: 15,000원
최종 가격: 135,000원

[5] AIMessage
내용:
**아이폰 17 가격 안내**

- **원래 가격**: 150,000원  
- **할인율**: 10%  
- **할인 금액**: 15,000원  
- **할인 적용 후 최종 가격**: **135,000원**

위 금액은 현재 기준으로, 재고 혹은 프로모션에 따라 변동될 수 있으니 구매 전에 다시 한 번 확인해 주세요.
최종 답변
**아이폰 17 가격 안내**

- **원래 가격**: 150,000원  
- **할인율**: 10%  
- **할인 금액**: 15,000원  
- **할인 적용 후 최종 가격**: **135,000원**

위 금액은 현재 기준으로, 재고 혹은 프로모션에 따라 변동될 수 있으니 구매 전에 다시 한 번 확인해 주세요.


In [13]:
import os

from typing import Annotated
from typing_extensions import TypedDict

from dotenv import load_dotenv

from langchain_nvidia_ai_endpoints import ChatNVIDIA

from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
)

from langgraph.graph import (
    StateGraph,
    START,
    END,
)

from langgraph.graph.message import add_messages

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600, max_tokens=2048)


class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def first_node(state: State):
    print("FIRST NODE")

    response = llm.invoke("안녕하세요라고 간단하게 인사해주세요.")

    print("생성된 메세지 : ", response.content)

    return {"messages": [response]}


def second_node(state: State):
    print("SECOND NODE")

    response = llm.invoke("자기소개를 아주 간단하게 해주세요")

    print("생성된 메세지 : ", response.content)

    return {"messages": [response]}


graph = StateGraph(State)

graph.add_node("first", first_node)

graph.add_node("second", second_node)

graph.add_edge(START, "first")
graph.add_edge("first", "second")
graph.add_edge("second", END)

app = graph.compile()

result = app.invoke({"messages": [HumanMessage(content="대화를 시작해주세요.")]})

print("\n\n")
print("=" * 60)
print("최종 State")
print("=" * 60)

for index, message in enumerate(result["messages"]):
    print(f"\n[{index}]")
    print(f"Type: {type(message).__name__}")
    print(f"Content: {message.content}")

/var/folders/cn/n1wp5rkx27j7415_2rb44sk00000gn/T/ipykernel_33054/1393133750.py:28: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600, max_tokens=2048)


FIRST NODE
생성된 메세지 :  안녕하세요
SECOND NODE
생성된 메세지 :  안녕하세요! 저는 OpenAI에서 만든 언어 모델, ChatGPT입니다. 기계 학습과 자연어 처리를 통해 여러분의 질문에 답하고 도움을 드리도록 설계되었습니다.



최종 State

[0]
Type: HumanMessage
Content: 대화를 시작해주세요.

[1]
Type: AIMessage
Content: 안녕하세요

[2]
Type: AIMessage
Content: 안녕하세요! 저는 OpenAI에서 만든 언어 모델, ChatGPT입니다. 기계 학습과 자연어 처리를 통해 여러분의 질문에 답하고 도움을 드리도록 설계되었습니다.


In [1]:
import os

from typing import Annotated
from typing_extensions import TypedDict

from dotenv import load_dotenv

from langchain_nvidia_ai_endpoints import ChatNVIDIA

from langgraph.graph import (
    StateGraph,
    START,
    END,
)


load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600, max_tokens=2048)


def merge_logs(old_logs: list[str], new_logs: list[str]) -> list[str]:
    merged = old_logs + new_logs

    return list(dict.fromkeys(merged))


class State(TypedDict):
    question: str
    answer: str
    logs: Annotated[list[str], merge_logs]


def search_node(state: State):
    print("search_node")
    response = llm.invoke(
        f"""
        다음 질문에 대해 핵심 정보를 정리해주세요.

        질문:
        {state["question"]}
        """
    )

    print(response.content)

    return {
        "answer": response.content,
        "logs": [
            "검색 시작",
            "검색 완료",
        ],
    }


def analyze_node(state: State):
    print("analyze_node")

    response = llm.invoke(
        f"""
        다음 정보를 분석해주세요.

        원래 질문:
        {state["question"]}

        검색 결과:
        {state["answer"]}
        """
    )

    print(response.content)

    return {"answer": response.content, "logs": ["분석 시작", "분석 완료"]}


def answer_node(state: State):
    print("answer_node")

    response = llm.invoke(
        f"""
        사용자의 질문에 최종 답변을 작성해주세요.

        질문:
        {state["question"]}

        분석 결과:
        {state["answer"]}
        """
    )

    print(response.content)

    return {"answer": response.content, "logs": ["답변 생성"]}


graph = StateGraph(State)

graph.add_node("search", search_node)
graph.add_node("analyze", analyze_node)
graph.add_node("answer", answer_node)

graph.add_edge(START, "search")

graph.add_edge("search", "analyze")

graph.add_edge("analyze", "answer")

graph.add_edge("answer", END)

app = graph.compile()

result = app.invoke(
    {
        "question": "LangGraph에서 Reducer는 왜 필요한가요?",
        "answer": "",
        "logs": [],
    }
)

print("최종 State")
print("=" * 60)

print("\n질문:")
print(result["question"])

print("\n최종 답변:")
print(result["answer"])

print("\n실행 로그:")
for index, log in enumerate(result["logs"], start=1):
    print(f"{index}. {log}")

/var/folders/cn/n1wp5rkx27j7415_2rb44sk00000gn/T/ipykernel_3960/3362362343.py:22: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600, max_tokens=2048)


search_node
### LangGraph에서 **Reducer**가 꼭 필요한 이유

| # | 핵심 포인트 | 설명 |
|---|-------------|------|
| 1 | **상태 합성(Synthesis)** | 여러 노드(시스템, LLM, 함수 등)의 출력물(`Observations`)을 하나의 구조화된 **state**로 합치기 위해 필요합니다.  |
| 2 | **검색/컨텍스트 축소** | 대규모 문헌, 로그, 메모리에서 필요한 정보만 추출·요약해 주어 기억 공간(메모리)를 최소화합니다.  |
| 3 | **조건부 흐름 제어** | 현재 state와 incoming data를 기준으로 다음 노드(Edge)를 동적으로 선택하거나 중단, 리피트 여부를 결정합니다. |
| 4 | **인풋 정규화** | 원시 텍스트/JSON을 정제, 포맷팅, 타입 변환해 다음 단계가 예측 가능한 형태로 입력받도록 보장합니다. |
| 5 | **예산/리소스 관리** | 토큰 사용량, 호출 횟수, 비용 등을 모니터링하며 필요 시 “줄임(Reducer)” 시켜서 비용을 절감합니다. |
| 6 | **결과/예측 통합** | 다중 서브프롬프트(예: 여러 LLM 답변)에서 얻은 정보를 종합해 최종 결과를 산출·정리합니다. |
| 7 | **로깅 및 추적** | 실행 단계마다 상태 변화를 기록해 디버깅/모니터링 보강.  |
| 8 | **다중 에이전트/작업 조율** | 여러 대리인(Agents) 또는 서브-작업이 동시에 수행될 때 결과를 한정된 상태에 반영해 전반적 흐름을 정리합니다. |

> **핵심**  
> Reducer는 “중간 생성물”을 **얻은 그대로 놓지 말고, 의미 있는 정보**만 남겨 정리해 최종 결과를 **효율적**이고 **예측 가능**하게 만드는 필수 도구입니다. LangGraph이 복잡한 워크플로우와 대규모 데이터, 멀티-스텝 추론을 관리할 때 Reducer 없이는 상태가 무질서해지고 리소스가 낭비됩니다.
analyze_node
## 1️⃣ Reducer

In [9]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver

from const.const import API_KEY, MODEL, EMBEDDING_MODEL

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600, max_completion_tokens=2048)


def chatbot(state: MessagesState):
    """현재 State에 들어있는 모든 messages를 LLM에게 전달합니다."""

    response = llm.invoke(state["messages"])

    return {"messages": [response]}


graph = StateGraph(MessagesState)

graph.add_node("chatbot", chatbot)

graph.add_edge(START, "chatbot")

graph.add_edge("chatbot", END)

checkpointer = InMemorySaver()

app = graph.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "user_1"}}

result1 = app.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 김민수야."}]}, config=config
)

print(result1["messages"][-1].content)

result2 = app.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐노"}]}, config=config
)

print(result2["messages"][-1].content)

current_state = app.get_state(config)

for message in current_state:
    # print(f"{message.type}: {message.content}")
    print("message : ", message)
    print()


config_2 = {"configurable": {"thread_id": "user_2"}}

result3 = app.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐라고 했냐"}]},
    config=config_2,
)

print(result3["messages"][-1].content)

네, 김민수님! 오늘은 무엇을 도와드릴까요? 궁금한 점이나 필요한 정보가 있으면 언제든 말씀해 주세요.
네, 김민수님! 지금 고민하고 있는 주제나 필요한 정보가 있으신가요? 언제든 말씀해 주세요!
message :  {'messages': [HumanMessage(content='내 이름은 김민수야.', additional_kwargs={}, response_metadata={}, id='4d8f50bb-cad1-49c3-8119-054d19173539'), AIMessage(content='네, 김민수님! 오늘은 무엇을 도와드릴까요? 궁금한 점이나 필요한 정보가 있으면 언제든 말씀해 주세요.', additional_kwargs={'reasoning_content': 'The user said "내 이름은 김민수야." (My name is Kim Minsoo). Likely want acknowledgement. We can respond in Korean, confirm. Maybe ask how can help.', 'reasoning': 'The user said "내 이름은 김민수야." (My name is Kim Minsoo). Likely want acknowledgement. We can respond in Korean, confirm. Maybe ask how can help.', '_reasoning_api_fields': ['reasoning_content', 'reasoning']}, response_metadata={'role': 'assistant', 'content': '네, 김민수님! 오늘은 무엇을 도와드릴까요? 궁금한 점이나 필요한 정보가 있으면 언제든 말씀해 주세요.', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': 'The user said "내 이름은 김민수야." (My name is